In [1]:
import numpy as np
from typing import Tuple, Optional

import jax
import jax.numpy as jnp
from jaxtyping import Array, Float, Int
import equinox as eqx

from phd.feature_search.jax_core.utils import tree_replace
from phd.feature_search.jax_core.tasks.geoff import NonlinearGEOFFTask, InputChangingGEOFFTask, BinaryRegressionTask

In [2]:
task = InputChangingGEOFFTask(
    n_features = 10,
    n_outputs = 10,
    flip_rate = 0.0,
    n_layers = 2,
    n_stationary_layers = 0,
    hidden_dim = 20,
    weight_scale = 1.0,
    activation = "ltu",
    sparsity = 0.0,
    weight_init = "binary",
    input_bounds = [-1.0, 1.0],
    input_subspace_range = 0.5,
    input_change_freq = 1, # 40_000,
    max_input_center_change = 2.0,
    # seed = 2512161,
)

pass_fracs = []
for i in range(30):
    task, (x, y) = task.generate_batch(batch_size=100)
    y_np = np.asarray(y)
    if y_np.ndim == 1:
        y_np = y_np[:, None]
    mean = y_np.mean(axis=0)
    pct10 = np.percentile(y_np, 10, axis=0)
    pct90 = np.percentile(y_np, 90, axis=0)
    
    pass_frac = np.mean([p10 <= 0 and p90 > 0 for p10, p90 in zip(pct10, pct90)])
    pass_fracs.append(pass_frac)

    # Print all output dimensions in a single row
    # print(
    #     " ".join(
    #         f"({p10:.2f}, {p90:.2f})"
    #         for p10, p90 in zip(pct10, pct90)
    #     )
    # )
    print(
        " ".join(
            ('T' if p10 <= 0 and p90 > 0 else '_')
            for p10, p90 in zip(pct10, pct90)
        )
    )

print(f"Pass Fraction: {np.mean(pass_fracs):.3f}")


_ T _ _ T _ T T _ T
T T _ T _ T _ T _ T
_ _ _ _ T T _ T _ _
_ T _ _ T T T T _ T
_ T _ _ _ _ _ T _ T
T T _ _ T T _ _ _ T
T T _ _ T _ T T _ _
T T _ _ T T T T _ _
T T _ _ _ T _ _ _ _
T T _ _ _ T _ T _ T
T T _ T T _ _ _ T T
T T _ _ _ T T T T T
_ T _ _ _ _ _ T _ T
T _ _ _ T T T T _ T
_ _ _ _ T T _ T _ _
T T _ T _ T _ _ _ _
T _ _ _ _ _ _ _ _ _
T _ _ _ _ _ _ T _ T
_ T _ _ _ T _ T _ _
T _ _ _ T _ _ T _ _
T T _ _ _ _ _ _ _ T
_ T _ _ _ _ _ T _ _
T T _ _ _ T _ T _ _
T T _ _ T T _ _ _ T
_ _ _ _ T _ T T _ _
T T _ _ T _ _ T _ _
_ T _ _ _ T _ _ _ T
T T _ _ T T _ T _ T
T _ _ _ _ T T T _ _
T T _ _ _ T _ T _ T
Pass Fraction: 0.417


In [43]:
task = BinaryRegressionTask(
    n_features = 10,
    n_outputs = 5,
    flip_rate = 0.0,
    n_layers = 4,
    n_stationary_layers = 0,
    hidden_dim = 10,
    weight_scale = 1.0,
    sparsity = 0.0,
    input_bounds = [-1.0, 1.0],
    input_subspace_range = 0.5,
    input_change_freq = 1, # 40_000,
    max_input_center_change = 0.1,
    # seed = 2512161,
)

pass_fracs = []
output_ranges = []
for i in range(30):
    task, (x, y) = task.generate_batch(batch_size=100)
    y_np = np.asarray(y)
    if y_np.ndim == 1:
        y_np = y_np[:, None]
    mean = y_np.mean(axis=0)
    pct10 = np.percentile(y_np, 10, axis=0)
    pct90 = np.percentile(y_np, 90, axis=0)
    
    pass_frac = np.mean([p10 <= 0 and p90 > 0 for p10, p90 in zip(pct10, pct90)])
    pass_fracs.append(pass_frac)

    # Print all output dimensions in a single row
    # print(
    #     " ".join(
    #         f"({p10:.2f}, {p90:.2f})"
    #         for p10, p90 in zip(pct10, pct90)
    #     )
    # )
    # print(
    #     " ".join(
    #         ('T' if p10 <= 0 and p90 > 0 else '_')
    #         for p10, p90 in zip(pct10, pct90)
    #     )
    # )
    print(
        " ".join(str(f'{m:.2f}') for m in mean)
    )

print(f"Pass Fraction: {np.mean(pass_fracs):.3f}")


0.38 0.61 0.55 0.37 0.43
0.52 0.66 0.38 0.34 0.46
0.45 0.65 0.47 0.32 0.55
0.45 0.68 0.45 0.32 0.60
0.52 0.61 0.44 0.37 0.60
0.54 0.67 0.40 0.45 0.56
0.60 0.66 0.26 0.50 0.47
0.56 0.67 0.35 0.45 0.46
0.63 0.65 0.30 0.50 0.57
0.56 0.70 0.36 0.49 0.50
0.54 0.67 0.40 0.43 0.54
0.69 0.62 0.30 0.48 0.55
0.62 0.62 0.34 0.44 0.50
0.45 0.56 0.48 0.39 0.44
0.30 0.46 0.59 0.35 0.25
0.10 0.28 0.76 0.23 0.14
0.17 0.25 0.75 0.14 0.16
0.13 0.27 0.71 0.25 0.13
0.06 0.20 0.81 0.31 0.03
0.12 0.32 0.70 0.28 0.09
0.03 0.22 0.78 0.25 0.06
0.15 0.39 0.62 0.29 0.14
0.18 0.39 0.67 0.20 0.28
0.06 0.32 0.80 0.24 0.18
0.08 0.28 0.88 0.23 0.24
0.09 0.31 0.84 0.23 0.23
0.03 0.13 0.84 0.21 0.09
0.06 0.27 0.75 0.26 0.16
0.08 0.30 0.70 0.34 0.21
0.07 0.41 0.86 0.22 0.36
Pass Fraction: 0.913
